# alpha-blended-eaf: Colab Pipeline

**Before running:**
1. Set the runtime to GPU (Runtime → Change runtime type → GPU).
2. Run Cell 1 to mount Drive.
3. Run Cell 2 to set up the environment and datasets.
4. Run Cell 3 (dry run) to validate training end-to-end.
5. Run Cell 4 for the full Stage 1 sweep.
6. Run Cell 5 for the full Stage 2 sweep.
7. Run Cell 6 to aggregate results and generate summary plots.
8. Run Cell 7 to train the final model on the full Japan dataset.
9. Run Cell 8 to evaluate the final model on India and China unseen datasets.

Re-running any cell is safe. Most setup steps use skip-if-done guards.

In [1]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
# Cell 2 — Environment setup
# First run: ~20-30 min. Subsequent runs: ~2-3 min.
set -e
cd /content
if [ ! -d /content/alpha-blended-eaf/.git ]; then
    git clone https://github.com/mtroque/alpha-blended-eaf.git
else
    cd /content/alpha-blended-eaf
    git fetch origin
    git reset --hard origin/main
    cd /content
fi
bash /content/alpha-blended-eaf/colab_run.sh

In [ ]:
%%bash
# Cell 3 — Dry run (1 config x 1 fold x 2 epochs, ~10 min)
# Run this before the full sweep. Success = prints metrics + 'Stage 1 sweep complete.'
set -e
REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results
KFOLD=/content/src/Japan_kfold
python "${REPO}/runner/run_sweep.py" \
    --sweep-config  "${REPO}/configs/sweep_configs.yaml" \
    --kfold-dir     "${KFOLD}" \
    --results-dir   "${DRIVE}" \
    --repo-root     "${REPO}" \
    --dry-run

In [ ]:
%%bash
# Cell 4 — Full Stage 1 sweep (4 configs x 10 folds = 40 runs)
# Run ONLY after Cell 3 passes. Safe to interrupt and resume.
set -e
REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results
KFOLD=/content/src/Japan_kfold
python "${REPO}/runner/run_sweep.py" \
    --sweep-config  "${REPO}/configs/sweep_configs.yaml" \
    --kfold-dir     "${KFOLD}" \
    --results-dir   "${DRIVE}" \
    --repo-root     "${REPO}"  \
    --stage         "stage1"

In [ ]:
%%bash
# Cell 5 — Full Stage 2 sweep (3 configs x 10 folds = 30 runs)
# Run after completing Stage 1 and adjusting sweep_configs.yaml.
set -e
REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results
KFOLD=/content/src/Japan_kfold
python -u "${REPO}/runner/run_sweep.py" \
    --sweep-config  "${REPO}/configs/sweep_configs.yaml" \
    --kfold-dir     "${KFOLD}" \
    --results-dir   "${DRIVE}" \
    --repo-root     "${REPO}"  \
    --stage         "stage2"

In [ ]:
%%bash
# Cell 6 — Aggregate sweep results and generate plots

set -e

REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results

python -u "${REPO}/runner/aggregate_results.py" \
    --runs-csv "${DRIVE}/sweep_runs/all_runs.csv" \
    --output-dir "${DRIVE}/sweep_runs"

In [ ]:
%%bash
# Cell 7 — Final training on full Japan dataset
# Trains ONE final model using the selected best parameters.

set -e

REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results

python -u "${REPO}/runner/train_final.py" \
    --config "${REPO}/configs/final_eval.yaml" \
    --results-dir "${DRIVE}" \
    --repo-root "${REPO}"

In [ ]:
%%bash
# Cell 8 — Evaluate final model on unseen India and China test datasets

set -e

REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results

WEIGHTS="${DRIVE}/final_runs/final_eps0.3_r0.6/train/weights/best.pt"

python -u "${REPO}/runner/evaluate_unseen.py" \
    --config "${REPO}/configs/final_eval.yaml" \
    --weights "${WEIGHTS}" \
    --results-dir "${DRIVE}/unseen_eval"

## Sweep Results and Final Evaluation

Stage 1 and Stage 2 results are written to:

```text
MyDrive/Dissertation_Results/Results/sweep_runs/
```

Main outputs:

```text
all_runs.csv
summary_by_config.csv
map_vs_f1_by_config.png
per_fold_map50_by_config.png
mean_metrics_by_config.png
```

Final training outputs:

```text
final_runs/final_eps0.1_r0.6/
```

Unseen evaluation outputs:

```text
unseen_eval/unseen_results.csv
```

Recommended workflow:
1. Complete Stage 1 and Stage 2 sweeps.
2. Run aggregation to identify the best configuration.
3. Update `configs/final_eval.yaml` with the final selected parameters.
4. Train the final model on the full Japan dataset.
5. Evaluate on India and China unseen datasets.